<a href="https://colab.research.google.com/github/fabiopauli/Qwen3.5-colab/blob/main/LTX2_Server_L4_GPU_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎥 LTX-2 Video Generation Server (A100 40GB GPU)

Run **LTX-2.3** (Lightricks' DiT-based audio-video foundation model) on a **Google Colab A100 GPU** (40 GB VRAM)
as a **local FastAPI server**.

Uses the **DistilledPipeline** with **FP8 quantization** for fast inference.
Text encoding (Gemma 3 12B) runs on **CPU** to maximize VRAM for video generation.

---

## ⚙️ Prerequisites — Colab Secrets

Before running, add this secret in the Colab sidebar (🔑 icon → "Secrets"):

| Secret Name | Where to get it | What it does |
|---|---|---|
| `HF_TOKEN` | https://huggingface.co/settings/tokens | Downloads model weights from HuggingFace |

> **Tip:** Toggle "Notebook access" ON for the secret after adding it.

---

## 📋 How to use

1. Select **Runtime → Change runtime type → A100 GPU**
2. Add your HF_TOKEN secret (see above)
3. Run **Cell 1** twice — first run installs packages and auto-restarts the runtime; second run downloads models (~50 GB, takes ~10-15 min)
4. Run **Cell 2** — loads model and starts the local FastAPI server on port 8081
5. Run **Cell 3** to generate a test video, or send requests from other Colab cells

### Example request (from another Colab cell)
```python
import requests

resp = requests.post("http://127.0.0.1:8081/generate", json={
    "prompt": "A golden retriever playing in ocean waves at sunset, slow motion, cinematic",
    "seed": 42,
    "height": 512,
    "width": 768,
    "num_frames": 49
}, timeout=600)

with open("output.mp4", "wb") as f:
    f.write(resp.content)
print("Video saved!")
```

---

## 📝 Notes
- **Model:** LTX-2.3 22B Distilled + Spatial Upscaler x2 + Gemma 3 12B (Q4)
- **Pipeline:** DistilledPipeline (8 steps stage 1, 4 steps stage 2 — fastest)
- **Quantization:** FP8 cast (fits in 40 GB VRAM)
- **Text encoding:** Runs on CPU to maximize VRAM for generation
- **Output:** MP4 video with synchronized audio
- **Default resolution:** 512×768 → upscaled to 1024×1536 (two-stage)
- **Default frames:** 49 (~2 seconds at 24 fps)

In [ ]:
# =============================================================================
# CELL 1 — Install Dependencies + Download Models
# =============================================================================
# Run this cell — it will install packages, restart the runtime automatically,
# then re-run to download models. Just run it once and wait.
# =============================================================================

import os, subprocess, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

_INSTALL_DONE = "/content/.ltx2_installed"

if not os.path.exists(_INSTALL_DONE):
    # ── Phase 1: Install packages, then restart runtime ──
    print("🔧 Phase 1/2 — Installing packages...")
    subprocess.run(["rm", "-rf", "/content/LTX-2"], check=True)
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/Lightricks/LTX-2.git", "/content/LTX-2"], check=True)

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.52,<4.58"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "-e", "/content/LTX-2/packages/ltx-core",
        "-e", "/content/LTX-2/packages/ltx-pipelines"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "fastapi", "uvicorn", "huggingface_hub", "hf_transfer"])

    # Mark install done so Phase 2 runs after restart
    open(_INSTALL_DONE, "w").close()

    print("✅ Packages installed. Restarting runtime to load new transformers...")
    os.kill(os.getpid(), 9)

else:
    # ── Phase 2: After restart — verify + download models ──
    print("🔧 Phase 2/2 — Downloading models...")
    import transformers
    print(f"   transformers version: {transformers.__version__}")

    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    from google.colab import userdata
    from huggingface_hub import hf_hub_download, snapshot_download

    HF_TOKEN = userdata.get("HF_TOKEN")
    MODEL_DIR = "/content/models"
    os.makedirs(MODEL_DIR, exist_ok=True)

    print("📦 Downloading LTX-2.3 distilled checkpoint...")
    hf_hub_download(
        repo_id="Lightricks/LTX-2.3",
        filename="ltx-2.3-22b-distilled.safetensors",
        local_dir=MODEL_DIR, token=HF_TOKEN,
    )
    print("📦 Downloading spatial upscaler (x2)...")
    hf_hub_download(
        repo_id="Lightricks/LTX-2.3",
        filename="ltx-2.3-spatial-upscaler-x2-1.0.safetensors",
        local_dir=MODEL_DIR, token=HF_TOKEN,
    )
    print("📦 Downloading Gemma 3 text encoder (Q4)...")
    snapshot_download(
        repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized",
        local_dir=os.path.join(MODEL_DIR, "gemma-3-12b-it-qat-q4_0-unquantized"),
        token=HF_TOKEN,
    )
    print("\n✅ All models downloaded! Proceed to Cell 2.")

In [ ]:
# =============================================================================
# CELL 2 — Load Model + Start FastAPI Server (Local)
# =============================================================================

import os, time, threading, tempfile, uuid, logging, subprocess, sys, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field
from google.colab import userdata

from ltx_core.quantization import QuantizationPolicy
from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
from ltx_pipelines.distilled import DistilledPipeline
from ltx_pipelines.utils.media_io import encode_video
from ltx_pipelines.utils import ModelLedger
from ltx_core.text_encoders.gemma.embeddings_processor import EmbeddingsProcessorOutput

logging.getLogger().setLevel(logging.INFO)

# ── Patch: run entire prompt encoding on CPU (frees VRAM for generation) ──
def _cpu_encode_prompts(prompts, model_ledger, **kwargs):
    """Encode prompts entirely on CPU, return results on CUDA."""
    cpu = torch.device("cpu")

    print("   Loading Gemma text encoder on CPU...")
    text_encoder = model_ledger.text_encoder_builder.build(device=cpu, dtype=model_ledger.dtype).eval()
    print("   ✅ Text encoder loaded on CPU")

    raw_outputs = [text_encoder.encode(p) for p in prompts]
    del text_encoder
    gc.collect()

    embeddings_processor = model_ledger.embeddings_processor_builder.build(device=cpu, dtype=model_ledger.dtype).eval()
    results = [embeddings_processor.process_hidden_states(hs, mask) for hs, mask in raw_outputs]
    del embeddings_processor, raw_outputs
    gc.collect()

    dev = model_ledger.device
    return [
        EmbeddingsProcessorOutput(
            video_encoding=r.video_encoding.to(dev),
            audio_encoding=r.audio_encoding.to(dev) if r.audio_encoding is not None else None,
            attention_mask=r.attention_mask.to(dev),
        )
        for r in results
    ]

import ltx_pipelines.utils.helpers as _helpers
import ltx_pipelines.distilled as _distilled_mod
_helpers.encode_prompts = _cpu_encode_prompts
_distilled_mod.encode_prompts = _cpu_encode_prompts
print("✅ Patched prompt encoding to run on CPU")

# ── Config ──
MODEL_DIR = "/content/models"
DISTILLED_CKPT = os.path.join(MODEL_DIR, "ltx-2.3-22b-distilled.safetensors")
UPSAMPLER_PATH = os.path.join(MODEL_DIR, "ltx-2.3-spatial-upscaler-x2-1.0.safetensors")
GEMMA_ROOT = os.path.join(MODEL_DIR, "gemma-3-12b-it-qat-q4_0-unquantized")
OUTPUT_DIR = "/content/outputs"
API_PORT = 8081

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Load the pipeline with FP8 quantization ──
print("🔄 Loading LTX-2.3 DistilledPipeline with FP8 quantization...")

pipeline = DistilledPipeline(
    distilled_checkpoint_path=DISTILLED_CKPT,
    spatial_upsampler_path=UPSAMPLER_PATH,
    gemma_root=GEMMA_ROOT,
    loras=[],
    quantization=QuantizationPolicy.fp8_cast(),
)

# Free VRAM after init — models are lazily loaded during inference
torch.cuda.empty_cache()
gc.collect()
print(f"✅ Pipeline loaded! GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

# ── 2. FastAPI app ──
app = FastAPI(title="LTX-2 Video Generation Server")

generation_lock = threading.Lock()


class GenerateRequest(BaseModel):
    prompt: str = Field(..., description="Text prompt describing the video to generate")
    seed: int = Field(default=42, description="Random seed for reproducibility")
    height: int = Field(default=512, description="Video height (will be 2x upscaled). Must be divisible by 32")
    width: int = Field(default=768, description="Video width (will be 2x upscaled). Must be divisible by 32")
    num_frames: int = Field(default=49, description="Number of frames. Must be 8*k+1 (e.g. 25, 49, 97)")
    frame_rate: float = Field(default=24.0, description="Frame rate (fps)")


@app.get("/health")
async def health():
    free = torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()
    return {
        "status": "ok",
        "model": "LTX-2.3-22B-Distilled",
        "quantization": "fp8-cast",
        "gpu": torch.cuda.get_device_name(0),
        "vram_free_gb": round(free / 1e9, 1),
    }


@app.post("/generate")
async def generate(req: GenerateRequest):
    if not generation_lock.acquire(blocking=False):
        return JSONResponse(
            status_code=503,
            content={"error": "A generation is already in progress. Please wait and retry."},
        )
    try:
        video_id = str(uuid.uuid4())[:8]
        output_path = os.path.join(OUTPUT_DIR, f"{video_id}.mp4")

        print(f"\n🎥 Generating video {video_id}...")
        print(f"   Prompt: {req.prompt[:80]}{'...' if len(req.prompt) > 80 else ''}")
        print(f"   Resolution: {req.width}x{req.height} -> {req.width*2}x{req.height*2} (upscaled)")
        print(f"   Frames: {req.num_frames} @ {req.frame_rate} fps")

        # Clear CUDA cache before generation
        torch.cuda.empty_cache()
        gc.collect()

        start_time = time.time()

        with torch.inference_mode():
            tiling_config = TilingConfig.default()
            video_chunks_number = get_video_chunks_number(req.num_frames, tiling_config)

            video, audio = pipeline(
                prompt=req.prompt,
                seed=req.seed,
                height=req.height * 2,
                width=req.width * 2,
                num_frames=req.num_frames,
                frame_rate=req.frame_rate,
                images=[],
                tiling_config=tiling_config,
            )

            encode_video(
                video=video,
                fps=req.frame_rate,
                audio=audio,
                output_path=output_path,
                video_chunks_number=video_chunks_number,
            )

        # Free generation tensors
        del video, audio
        torch.cuda.empty_cache()
        gc.collect()

        elapsed = time.time() - start_time
        print(f"✅ Video generated in {elapsed:.1f}s -> {output_path}")

        return FileResponse(
            output_path,
            media_type="video/mp4",
            filename=f"ltx2_{video_id}.mp4",
            headers={"X-Generation-Time": f"{elapsed:.1f}s"},
        )
    except Exception as e:
        logging.exception("Generation failed")
        torch.cuda.empty_cache()
        gc.collect()
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        generation_lock.release()


# ── 3. Start local server ──
server_thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": API_PORT, "log_level": "warning"},
    daemon=True,
)
server_thread.start()
time.sleep(2)

print("\n" + "=" * 65)
print(f"🏠 LOCAL API URL:  http://127.0.0.1:{API_PORT}")
print(f"=" * 65)
print(f"\n  GET  http://127.0.0.1:{API_PORT}/health")
print(f"  POST http://127.0.0.1:{API_PORT}/generate")
print(f"  GET  http://127.0.0.1:{API_PORT}/docs   (interactive Swagger UI)")
print(f"\n✅ Server is running! Send requests to generate videos.")

In [ ]:
# =============================================================================
# CELL 3 — Generate a Video (Quick Test)
# =============================================================================

import requests
from IPython.display import Video, display

API_URL = "http://127.0.0.1:8081/generate"

print("🎥 Generating test video...")
resp = requests.post(API_URL, json={
    "prompt": (
        "A golden retriever puppy runs along a sandy beach at sunset. "
        "Waves crash gently in the background, spraying mist into warm golden light. "
        "The puppy's fur glows in the low sun as it splashes through shallow water. "
        "Camera follows at a low angle, tracking the dog in slow motion. "
        "Cinematic, warm color grading, shallow depth of field."
    ),
    "seed": 42,
    "height": 512,
    "width": 768,
    "num_frames": 49,
    "frame_rate": 24.0,
}, timeout=600)

if resp.status_code == 200:
    output_file = "/content/outputs/test_video.mp4"
    with open(output_file, "wb") as f:
        f.write(resp.content)
    gen_time = resp.headers.get("X-Generation-Time", "unknown")
    print(f"✅ Video saved to {output_file} (generation time: {gen_time})")
    display(Video(output_file, embed=True, width=640))
else:
    print(f"❌ Error: {resp.status_code} - {resp.text}")

In [ ]:
# =============================================================================
# CELL 4 — Image-to-Video Generation (Optional)
# =============================================================================
# Upload an image to Colab, then use it as conditioning for video generation.
#
# To use: upload an image via the Colab file browser, then set IMAGE_PATH below.

import requests, base64, json
from IPython.display import Video, display
from google.colab import files

# Upload an image
print("📁 Upload an image to use as conditioning:")
uploaded = files.upload()

if uploaded:
    IMAGE_PATH = "/content/" + list(uploaded.keys())[0]
    print(f"\n🖼️ Using image: {IMAGE_PATH}")
    print("🎥 Generating image-conditioned video...")

    # For image-to-video, we need to use the CLI directly since the
    # DistilledPipeline accepts images as ImageConditioningInput tuples.
    # Using subprocess to call the pipeline with image conditioning:
    import subprocess
    result = subprocess.run([
        "python", "-m", "ltx_pipelines.distilled",
        "--distilled-checkpoint-path", "/content/models/ltx-2.3-22b-distilled.safetensors",
        "--spatial-upsampler-path", "/content/models/ltx-2.3-spatial-upscaler-x2-1.0.safetensors",
        "--gemma-root", "/content/models/gemma-3-12b-it-qat-q4_0-unquantized",
        "--quantization", "fp8-cast",
        "--prompt", "A cinematic video inspired by the input image, smooth camera motion, high quality",
        "--image", IMAGE_PATH, "0", "1.0",
        "--output-path", "/content/outputs/i2v_output.mp4",
        "--seed", "42",
        "--num-frames", "97",
    ], capture_output=True, text=True,
       env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})

    print(result.stdout[-500:] if result.stdout else "")
    if result.returncode == 0:
        print("✅ Image-to-video complete!")
        display(Video("/content/outputs/i2v_output.mp4", embed=True, width=640))
    else:
        print(f"❌ Error: {result.stderr[-500:]}")
else:
    print("No image uploaded. Skipping.")